# Pretrained Model Challenge — Electronics Reviews (Independent Build)

Same brief as the class "Pretrained Model Challenge" (sentiment comparison across models + topic
classification), built independently with:
- a different synthetic dataset (e-commerce electronics reviews instead of restaurant/hotel reviews)
- three different pretrained sentiment models
- a different technique for the topic task: sentence-embedding similarity instead of zero-shot NLI


## Setup

In [ ]:
!pip install -q transformers sentence-transformers


In [ ]:
import pandas as pd
from transformers import pipeline
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report


## Task A: Sentiment — Synthetic Electronics Reviews

In [ ]:
sentiment_rows = [('This wireless charger juices my phone up faster than the one it came with.', 'positive'), ('Battery easily lasts two full days of heavy use, way better than my old earbuds.', 'positive'), ('Setup took less than two minutes and it paired with three devices without a hitch.', 'positive'), ('Sound quality on these headphones is shockingly good for the price.', 'positive'), ('Customer support replaced my unit within 48 hours, no questions asked.', 'positive'), ('The build feels solid, no creaks or loose parts anywhere.', 'positive'), ('Screen is bright even in direct sunlight, perfect for outdoor use.', 'positive'), ("Noise cancellation actually works, I can't hear my neighbor's construction anymore.", 'positive'), ('Shipping was fast, arrived two days early in perfect condition.', 'positive'), ('This laptop stand folds flat and fits right in my backpack.', 'positive'), ('Battery life on the smartwatch is incredible, barely charge it once a week.', 'positive'), ('The app is intuitive, I had it configured in under five minutes.', 'positive'), ('Great value, works just as well as brands twice the price.', 'positive'), ('Case feels premium, definitely not what I expected for this price point.', 'positive'), ('The mic quality on calls is crystal clear, coworkers stopped complaining.', 'positive'), ('Returned a defective unit and got a replacement with zero hassle.', 'positive'), ('Keyboard has a satisfying click without being too loud.', 'positive'), ('Watch face is easy to read even without my glasses on.', 'positive'), ('Charging cable is thick and durable, no fraying after months of daily use.', 'positive'), ("Best budget monitor I've used, colors are surprisingly accurate.", 'positive'), ('Fits comfortably for hours, no ear fatigue even during long calls.', 'positive'), ('Setup guide was clear and the whole thing worked first try.', 'positive'), ('Impressed by how quiet the fan is under heavy load.', 'positive'), ('Packaging was eco-friendly and the product itself exceeded expectations.', 'positive'), ('Support team walked me through the fix over chat in ten minutes.', 'positive'), ('Battery died after six months, barely holds a charge now.', 'negative'), ('Bluetooth keeps disconnecting every few minutes, unusable for calls.', 'negative'), ('Arrived with a cracked screen, clearly not packaged properly.', 'negative'), ('Customer support took three weeks to even respond to my email.', 'negative'), ('The charging port stopped working after two weeks of normal use.', 'negative'), ('Build quality feels cheap, the hinge already wobbles.', 'negative'), ('Description said waterproof, mine died the first time it got splashed.', 'negative'), ('App constantly crashes and loses my saved settings.', 'negative'), ("Ordered the black one, received white, and support won't fix it.", 'negative'), ('Fan makes a loud grinding noise within the first week.', 'negative'), ('Screen has noticeable dead pixels right out of the box.', 'negative'), ('Instructions were in the wrong language, no English manual included.', 'negative'), ('Battery indicator is wildly inaccurate, dies with 40% still showing.', 'negative'), ('Return process was a nightmare, took over a month to get refunded.', 'negative'), ('Headphones cut out constantly, completely unusable during workouts.', 'negative'), ('Case cracked the first time I dropped it from waist height.', 'negative'), ("The 'fast charger' takes longer than my old one ever did.", 'negative'), ('Wow, love paying full price for a product that stops working in a week.', 'negative'), ('Meh, does the job I guess.', 'negative'), ('5 stars if you enjoy troubleshooting instead of actually using the product.', 'negative'), ("Great sound, if you don't mind it cutting out every ten seconds.", 'negative'), ('Shipping was fast, too bad the product barely works.', 'negative'), ("It's fine, nothing special, wouldn't buy again.", 'negative'), ('First charge and last charge, RIP.', 'negative'), ('Works okay for a day, then randomly stops pairing.', 'negative'), ('The design is beautiful, shame the battery only lasts an hour.', 'negative'), ("Not bad for the price, though I've used better.", 'positive'), ("Customer service was polite while telling me they couldn't help at all.", 'negative'), ("I don't usually leave reviews, but this thing earned one.", 'positive'), ('10/10 would not recommend to my worst enemy.', 'negative')]

df = pd.DataFrame(sentiment_rows, columns=["text", "true_label"])
print(df.shape)
df.head()


## Task A: 3 sentiment models

Three different pretrained models/orgs — a general-purpose binary model, an informal/social-text model, and an emotion model repurposed for polarity (a genuinely different label space to normalize).

In [ ]:
sentiment_model_ids = [
    "siebert/sentiment-roberta-large-english",
    "finiteautomata/bertweet-base-sentiment-analysis",
    "j-hartmann/emotion-english-distilroberta-base",
]

sentiment_pipelines = {
    model_id: pipeline("text-classification", model=model_id)
    for model_id in sentiment_model_ids
}


## Label Normalization

Each model returns labels in a different format:
- `siebert`: `POSITIVE` / `NEGATIVE`
- `bertweet`: `POS` / `NEG` / `NEU`
- `emotion model`: `joy`, `anger`, `sadness`, `fear`, `disgust`, `surprise`, `neutral`

Rather than a lookup table keyed on model ID, this normalizer reads the label text itself (a prefix
match plus an emotion→polarity map) so it would generalize to a new model without being told which
one it is.

In [ ]:
EMOTION_POLARITY = {
    "joy": "positive",
    "surprise": "positive",
    "anger": "negative",
    "disgust": "negative",
    "fear": "negative",
    "sadness": "negative",
    "neutral": "negative",  # treated as negative to keep a binary comparison against ground truth
}

def normalize_label(raw_label: str) -> str:
    label = raw_label.strip().upper()
    if label.startswith("POS"):
        return "positive"
    if label.startswith("NEG"):
        return "negative"
    if label.startswith("NEU"):
        return "negative"
    digits = "".join(ch for ch in label if ch.isdigit())
    if digits:
        return "positive" if int(digits) >= 4 else "negative"
    if label.lower() in EMOTION_POLARITY:
        return EMOTION_POLARITY[label.lower()]
    raise ValueError(f"Unrecognized label format: {raw_label}")


def predict_sentiment(text: str, model_id: str) -> dict:
    result = sentiment_pipelines[model_id](text)[0]
    return {
        "label": result["label"],
        "score": result["score"],
        "normalized_label": normalize_label(result["label"]),
        "metadata": "huggingface_AI_model",
    }


# quick sanity check on a few varied examples
samples = [
    "The battery on this thing is incredible, lasts me all week.",
    "Screen cracked out of the box and support went silent.",
    "Wow, love paying full price for a product that stops working in a week.",
]
for sample in samples:
    print(sample)
    for model_id in sentiment_model_ids:
        print(" ", model_id, "->", predict_sentiment(sample, model_id))
    print()


## Evaluate Each Model

In [ ]:
results = {}
for model_id in sentiment_model_ids:
    predictions = [predict_sentiment(t, model_id) for t in df["text"]]
    df[f"pred_{model_id}"] = [p["normalized_label"] for p in predictions]
    df[f"score_{model_id}"] = [p["score"] for p in predictions]

    acc = accuracy_score(df["true_label"], df[f"pred_{model_id}"])
    f1 = f1_score(df["true_label"], df[f"pred_{model_id}"], pos_label="positive")
    results[model_id] = {"accuracy": acc, "f1": f1}

    print(model_id)
    print(classification_report(df["true_label"], df[f"pred_{model_id}"]))
    print("confusion matrix [rows=true, cols=pred] (order: negative, positive):")
    print(confusion_matrix(df["true_label"], df[f"pred_{model_id}"], labels=["negative", "positive"]))
    print()

results_df = pd.DataFrame(results).T
results_df


## Error Analysis

In [ ]:
pred_cols = [f"pred_{m}" for m in sentiment_model_ids]
df["any_wrong"] = df[pred_cols].ne(df["true_label"], axis=0).any(axis=1)

errors = df[df["any_wrong"]][["text", "true_label"] + pred_cols]
print(f"{len(errors)} of {len(df)} examples had at least one wrong prediction")
errors


### Does confidence drop on the errors?

A different diagnostic from a plain right/wrong table: for each model, compare its average confidence
score on examples it got right vs. examples it got wrong. If confidence is noticeably lower on the
misses, low-confidence predictions are a usable signal for flagging uncertain calls; if it's about the
same, the model is "confidently wrong" and confidence alone won't catch its mistakes.

In [ ]:
confidence_summary = []
for model_id in sentiment_model_ids:
    pred_col, score_col = f"pred_{model_id}", f"score_{model_id}"
    is_correct = df[pred_col] == df["true_label"]
    confidence_summary.append({
        "model": model_id,
        "avg_confidence_when_correct": df.loc[is_correct, score_col].mean(),
        "avg_confidence_when_wrong": df.loc[~is_correct, score_col].mean(),
    })

pd.DataFrame(confidence_summary)


**Record at least 10 interesting failures here** (fill in after running the cells above). A few
examples worth watching, given what's in the dataset:

| Input | Expected | Likely failure mode |
|---|---|---|
| "Wow, love paying full price for a product that stops working in a week." | negative | sarcasm — "love" and "full price" read as positive signal |
| "5 stars if you enjoy troubleshooting instead of actually using the product." | negative | sarcastic use of "5 stars" |
| "Great sound, if you don't mind it cutting out every ten seconds." | negative | mixed sentiment, model may anchor on "Great sound" |
| "Not bad for the price, though I've used better." | positive | double negative + hedge, ambiguous polarity |
| "The design is beautiful, shame the battery only lasts an hour." | negative | leads with a positive clause before the real complaint |
| "Meh, does the job I guess." | negative | too short/flat for the model to find a strong signal |
| "10/10 would not recommend to my worst enemy." | negative | numeric score reads as positive despite negative context |


## Model Battle

Compare the three sentiment models on accuracy/F1 (above), plus:

| Model | Size | Label scheme | License | Best for |
|---|---|---|---|---|
| siebert/sentiment-roberta-large-english | Large (355M) | Binary POS/NEG | MIT | Robust general-purpose sentiment, less domain-fragile |
| finiteautomata/bertweet-base-sentiment-analysis | Base (135M) | POS/NEG/NEU | MIT | Informal/social text, has a neutral class |
| j-hartmann/emotion-english-distilroberta-base | Small (82M) | 7 emotions | MIT | Finer-grained emotional read, but needs a polarity mapping layer to answer a plain positive/negative question |

**Recommendation:** fill in based on `results_df` once you run this — a model that scores highest on
this small binary set isn't automatically the best pick if the real use case needs the neutral class
(bertweet) or a finer emotional read (the emotion model) rather than pure accuracy on positive/negative.

## Task B: Topic Classification — Embedding Similarity

Different technique from NLI-based zero-shot classification: instead of asking a model "does this text
entail this label," this embeds the review and a short description of each topic with a sentence-embedding
model, then assigns the topic whose description embedding is closest by cosine similarity. No NLI model
involved at all.

In [ ]:
topic_rows = [('Battery easily lasts two full days of heavy use.', 'Battery Life'), ('The build feels solid, no creaks or loose parts anywhere.', 'Build Quality'), ('Sound quality on these headphones is shockingly good for the price.', 'Audio Quality'), ('Customer support replaced my unit within 48 hours, no questions asked.', 'Customer Support'), ('Shipping was fast, arrived two days early in perfect condition.', 'Shipping & Packaging'), ('The app is intuitive, I had it configured in under five minutes.', 'Software/App'), ('Great value, works just as well as brands twice the price.', 'Price/Value'), ('Screen is bright even in direct sunlight.', 'Display/Screen'), ('Battery died after six months, barely holds a charge now.', 'Battery Life'), ('Bluetooth keeps disconnecting every few minutes.', 'Software/App'), ('Arrived with a cracked screen, clearly not packaged properly.', 'Shipping & Packaging'), ('Customer support took three weeks to even respond.', 'Customer Support'), ('The charging port stopped working after two weeks.', 'Build Quality'), ('Build quality feels cheap, the hinge already wobbles.', 'Build Quality'), ('App constantly crashes and loses my saved settings.', 'Software/App'), ('Fan makes a loud grinding noise within the first week.', 'Build Quality'), ('Screen has noticeable dead pixels right out of the box.', 'Display/Screen'), ('Battery indicator is wildly inaccurate.', 'Battery Life'), ('Return process was a nightmare, took over a month to refund.', 'Customer Support'), ('Headphones cut out constantly during workouts.', 'Audio Quality'), ('Case cracked the first time I dropped it.', 'Build Quality'), ("The 'fast charger' takes longer than my old one.", 'Battery Life'), ('Watch face is easy to read even without my glasses.', 'Display/Screen'), ('Charging cable is thick and durable, no fraying.', 'Build Quality'), ('Best budget monitor, colors are surprisingly accurate.', 'Display/Screen'), ('Fits comfortably for hours, no ear fatigue.', 'Audio Quality'), ('Setup guide was clear and worked first try.', 'Software/App'), ('Impressed by how quiet the fan is under heavy load.', 'Build Quality'), ('Packaging was eco-friendly and product exceeded expectations.', 'Shipping & Packaging'), ('Support team walked me through the fix over chat.', 'Customer Support'), ("Ordered black, received white, and support won't fix it.", 'Shipping & Packaging'), ('Instructions were in the wrong language, no English manual.', 'Shipping & Packaging'), ('Mic quality on calls is crystal clear.', 'Audio Quality'), ('Keyboard has a satisfying click.', 'Build Quality'), ("Great sound, if you don't mind it cutting out every ten seconds.", 'Audio Quality'), ('Price is fair for what you get, definitely worth it.', 'Price/Value')]

topic_df = pd.DataFrame(topic_rows, columns=["text", "true_topic"])

topic_descriptions = {
    "Battery Life": "comments about battery life, charging speed, or how long a charge lasts",
    "Build Quality": "comments about physical durability, materials, hinges, cracks, or manufacturing defects",
    "Audio Quality": "comments about sound quality, noise cancellation, or microphone clarity",
    "Customer Support": "comments about customer service, support response time, or replacements",
    "Shipping & Packaging": "comments about delivery speed, packaging condition, or receiving the wrong item",
    "Software/App": "comments about the companion app, firmware, pairing, or setup process",
    "Price/Value": "comments about price, cost, or value for money",
    "Display/Screen": "comments about screen brightness, resolution, dead pixels, or visibility",
}


In [ ]:
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

topic_names = list(topic_descriptions.keys())
topic_embeddings = embedder.encode(list(topic_descriptions.values()), convert_to_tensor=True)

def predict_topic(text: str) -> dict:
    text_embedding = embedder.encode(text, convert_to_tensor=True)
    similarities = util.cos_sim(text_embedding, topic_embeddings)[0]
    top2 = similarities.topk(2)
    best_idx = int(top2.indices[0])
    return {
        "label": topic_names[best_idx],
        "similarity": float(top2.values[0]),
        "gap_to_runner_up": float(top2.values[0] - top2.values[1]),
    }

topic_predictions = topic_df["text"].apply(predict_topic)
topic_df["pred_topic"] = topic_predictions.apply(lambda p: p["label"])
topic_df["similarity"] = topic_predictions.apply(lambda p: p["similarity"])
topic_df["gap_to_runner_up"] = topic_predictions.apply(lambda p: p["gap_to_runner_up"])
topic_df.head()


In [ ]:
topic_acc = accuracy_score(topic_df["true_topic"], topic_df["pred_topic"])
topic_f1 = f1_score(topic_df["true_topic"], topic_df["pred_topic"], average="macro")
print("Embedding-similarity topic accuracy:", round(topic_acc, 3), " macro-F1:", round(topic_f1, 3))

topic_errors = topic_df[topic_df["true_topic"] != topic_df["pred_topic"]]
print(f"{len(topic_errors)} of {len(topic_df)} misclassified")
# smallest gap first = the genuinely ambiguous calls, where two topics were nearly tied
topic_errors.sort_values("gap_to_runner_up")


## Does the Model Understand My Domain?

---

- The sentiment models were trained on movie reviews, tweets, or generic product reviews — not
  electronics-specific ones. Domain jargon like "hinge," "dead pixels," "pairing," and "firmware" is
  a narrower vocabulary than what most of these models saw in training, which is a domain-shift risk
  even before considering informal phrasing.
- The emotion model (`j-hartmann/emotion-english-distilroberta-base`) is answering a different
  question than the other two — it was never trained on "positive/negative," so its usefulness here
  depends entirely on whether the emotion→polarity mapping above is a reasonable proxy for the task.
- The embedding-similarity approach to topics has its own failure mode: it doesn't reason about the
  text, it just measures how close the review sits to each topic *description* in embedding space. That
  means classification quality is partly a function of how well-written the topic descriptions are, not
  just the review text — a lever the zero-shot NLI approach doesn't have.
- None of these models have seen mixed English/Arabic or Bahrain-specific product reviews, which is a
  gap regardless of which technique is used.

Considering the accuracy/F1 figures above:
- Use a pretrained model as-is if accuracy is high (roughly 80%+) and the failures are concentrated on
  genuinely ambiguous or sarcastic language — that's a hard problem for humans too.
- Fine-tuning on real electronics-review data would be worth it if errors concentrate on domain jargon
  the models weren't trained on, since that's a systematic gap rather than random noise.
- For the topic task specifically: if most misclassifications have a small `gap_to_runner_up`, that
  points to genuinely overlapping categories (e.g. "battery" vs. "build quality" defects) — the fix is
  sharpening the topic *descriptions*, not fine-tuning a bigger model.
